# Random Forest SHAP Analysis

This notebook explains the saved random-forest bundle without retraining the model.

It has three goals:
1. identify which features most strongly influence predicted price
2. generate a few clean plots for the thesis
3. decide which inputs are worth exposing in the UI

## Setup

The notebook loads the saved bundle from `artifacts/random_forest_final/full_data_bundle` and runs SHAP directly on that fitted model.

In [1]:
from pathlib import Path
import json
import os
import sys
from time import perf_counter
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import TqdmWarning
from IPython.display import Image, display

warnings.filterwarnings("ignore", category=TqdmWarning)

import shap

ROOT = Path.cwd().resolve()
if (ROOT / "src").exists():
    REPO_ROOT = ROOT
else:
    REPO_ROOT = ROOT.parent.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/mpl")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp")

from src.random_forest_serving import load_random_forest_bundle

BUNDLE_DIR = REPO_ROOT / "artifacts" / "random_forest_final" / "full_data_bundle"
OUTPUT_DIR = REPO_ROOT / "artifacts" / "random_forest_shap"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_SIZE = 600
RANDOM_STATE = 42
LOCAL_ROW_INDEX = 0

## Helper Functions

The code below keeps the later analysis cells short and readable.

In [2]:
def sample_reference_rows(bundle_dir, feature_names, sample_size, random_state):
    reference_rows = pd.read_csv(bundle_dir / "reference_rows.csv")
    feature_frame = reference_rows.loc[:, feature_names].copy()
    if feature_frame.shape[0] != min(feature_frame.shape[0], sample_size):
        feature_frame = feature_frame.sample(n=sample_size, random_state=random_state).reset_index(drop=True)
    return feature_frame


def to_dense_float_array(matrix):
    if hasattr(matrix, "toarray"):
        matrix = matrix.toarray()
    return np.asarray(matrix, dtype=float)


def raw_feature_name(transformed_name, raw_features):
    if transformed_name.startswith("num__"):
        return transformed_name.removeprefix("num__")

    if transformed_name.startswith("cat__"):
        remainder = transformed_name.removeprefix("cat__")
        matches = [
            feature
            for feature in raw_features
            if remainder == feature or remainder.startswith(feature + "_")
        ]
        if matches:
            return max(matches, key=len)
        return remainder

    return transformed_name


def group_shap_values(
    shap_values,
    transformed_feature_names,
    raw_feature_names,
    row_index,
):
    grouped_indices = {}
    for idx, transformed_name in enumerate(transformed_feature_names):
        grouped_name = raw_feature_name(transformed_name, raw_feature_names)
        grouped_indices.setdefault(grouped_name, []).append(idx)

    grouped_shap = pd.DataFrame(index=row_index)
    for feature_name, indices in grouped_indices.items():
        grouped_shap[feature_name] = shap_values[:, indices].sum(axis=1)

    return grouped_shap


def global_importance_table(grouped_shap):
    importance_df = pd.DataFrame(
        {
            "feature_name": grouped_shap.columns,
            "mean_abs_shap": grouped_shap.abs().mean(axis=0).to_numpy(),
            "mean_shap": grouped_shap.mean(axis=0).to_numpy(),
        }
    )
    return importance_df.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)


def local_explanation_table(feature_frame, grouped_shap, row_index, top_k=12):
    local_df = pd.DataFrame(
        {
            "feature_name": grouped_shap.columns,
            "feature_value": [feature_frame.iloc[row_index][column] for column in grouped_shap.columns],
            "shap_value": grouped_shap.iloc[row_index].to_numpy(),
        }
    )
    local_df["abs_shap_value"] = local_df["shap_value"].abs()
    return local_df.sort_values("abs_shap_value", ascending=False).head(top_k).reset_index(drop=True)


def save_global_importance_plot(importance_df, output_path, top_k=15):
    plot_df = importance_df.head(top_k).sort_values("mean_abs_shap", ascending=True)
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(plot_df["feature_name"], plot_df["mean_abs_shap"], color="#355c7d")
    ax.set_xlabel("Mean absolute SHAP value (EUR)")
    ax.set_ylabel("")
    ax.set_title("Top SHAP Drivers of Predicted Price")
    ax.grid(axis="x", alpha=0.2)
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return output_path


def save_dependence_grid(feature_frame, grouped_shap, importance_df, output_path, top_k=4):
    selected_features = [feature for feature in importance_df["feature_name"].tolist() if feature in feature_frame.columns][:top_k]
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    axes = axes.ravel()

    for ax, feature_name in zip(axes, selected_features):
        x = feature_frame[feature_name]
        y = grouped_shap[feature_name]

        if pd.api.types.is_numeric_dtype(x):
            ax.scatter(x, y, s=18, alpha=0.6, color="#c06c84")
            ax.set_xlabel(feature_name)
        else:
            top_categories = x.astype(str).value_counts().head(8).index.tolist()
            mask = x.astype(str).isin(top_categories)
            plot_df = pd.DataFrame({"feature_value": x.astype(str)[mask], "shap_value": y[mask]})
            order = plot_df.groupby("feature_value")["shap_value"].median().sort_values().index.tolist()
            grouped_values = [plot_df.loc[plot_df["feature_value"] == category, "shap_value"] for category in order]
            ax.boxplot(grouped_values, tick_labels=order, vert=True, patch_artist=True)
            ax.tick_params(axis="x", rotation=30)
            ax.set_xlabel(feature_name)

        ax.axhline(0.0, color="black", linewidth=1, alpha=0.4)
        ax.set_ylabel("SHAP value (EUR)")
        ax.set_title(feature_name)
        ax.grid(alpha=0.15)

    for ax in axes[len(selected_features):]:
        ax.axis("off")

    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return output_path


def save_local_explanation_plot(local_df, predicted_price, base_value, output_path):
    plot_df = local_df.sort_values("shap_value")
    colors = np.where(np.greater_equal(plot_df["shap_value"], 0), "#2a9d8f", "#e76f51")
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(plot_df["feature_name"], plot_df["shap_value"], color=colors)
    ax.axvline(0.0, color="black", linewidth=1, alpha=0.4)
    ax.set_xlabel("Contribution to price prediction (EUR)")
    ax.set_ylabel("")
    ax.set_title(f"Local SHAP Explanation | base={base_value:.2f} EUR | prediction={predicted_price:.2f} EUR")
    ax.grid(axis="x", alpha=0.2)
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return output_path

## Load Model And Compute SHAP Values

This cell prepares the sampled reference rows, applies the saved preprocessing pipeline, computes TreeSHAP values on the transformed matrix, and groups the transformed columns back to the original raw features.

In [ ]:
timings = []

step_start = perf_counter()
bundle = load_random_forest_bundle(BUNDLE_DIR)
timings.append(("load_bundle", perf_counter() - step_start))

feature_names = list(bundle["metadata"]["feature_names"])

step_start = perf_counter()
feature_frame = sample_reference_rows(BUNDLE_DIR, feature_names, SAMPLE_SIZE, RANDOM_STATE)
timings.append(("load_reference_rows", perf_counter() - step_start))

model_pipeline = bundle["model"]
preprocessor = model_pipeline.named_steps["preprocessor"]
forest = model_pipeline.named_steps["model"]

step_start = perf_counter()
transformed_matrix = to_dense_float_array(preprocessor.transform(feature_frame))
transformed_feature_names = list(preprocessor.get_feature_names_out())
timings.append(("preprocess_features", perf_counter() - step_start))

step_start = perf_counter()
explainer = shap.TreeExplainer(forest)
timings.append(("build_tree_explainer", perf_counter() - step_start))

step_start = perf_counter()
shap_values = np.asarray(explainer.shap_values(transformed_matrix), dtype=float)
base_value = float(np.asarray(explainer.expected_value).reshape(-1)[0])
timings.append(("compute_shap_values", perf_counter() - step_start))

step_start = perf_counter()
point_predictions = model_pipeline.predict(feature_frame)
timings.append(("predict_sample_rows", perf_counter() - step_start))

step_start = perf_counter()
grouped_shap = group_shap_values(
    shap_values=shap_values,
    transformed_feature_names=transformed_feature_names,
    raw_feature_names=feature_names,
    row_index=feature_frame.index,
)
timings.append(("group_shap_values", perf_counter() - step_start))

step_start = perf_counter()
importance_df = global_importance_table(grouped_shap)
local_df = local_explanation_table(feature_frame, grouped_shap, row_index=LOCAL_ROW_INDEX)
timings.append(("build_summary_tables", perf_counter() - step_start))

bundle_size_mb = round(
    sum(path.stat().st_size for path in BUNDLE_DIR.rglob("*") if path.is_file()) / (1024 ** 2),
    2,
)
timing_df = pd.DataFrame(timings, columns=["step", "seconds"])

pd.concat(
    [
        pd.DataFrame(
            {
                "sampled_rows": [len(feature_frame)],
                "raw_features": [len(feature_names)],
                "transformed_features": [len(transformed_feature_names)],
                "base_value_eur": [round(base_value, 2)],
                "bundle_size_mb": [bundle_size_mb],
            }
        ),
        timing_df.set_index("step").T.reset_index(drop=True),
    ],
    axis=1,
)

## Research Question: Which Factors Influence Price?

This table is the main direct answer. It shows which features have the largest average effect on the model output.

In [ ]:
importance_df.head(15)

## Plot 1: Global Feature Importance

In [ ]:
global_plot_path = save_global_importance_plot(
    importance_df=importance_df,
    output_path=OUTPUT_DIR / "top_shap_features.png",
)
display(Image(filename=str(global_plot_path)))

## Plot 2: Dependence Patterns

These plots help interpret directionality. For example, they show whether higher values tend to push predictions up or down.

In [ ]:
dependence_plot_path = save_dependence_grid(
    feature_frame=feature_frame,
    grouped_shap=grouped_shap,
    importance_df=importance_df,
    output_path=OUTPUT_DIR / "top_feature_dependence_grid.png",
)
display(Image(filename=str(dependence_plot_path)))

## Plot 3: Local Explanation Example

This plot is useful for the demo and the UI discussion because it shows one concrete prediction broken into its largest positive and negative contributors.

In [ ]:
local_prediction = float(point_predictions[LOCAL_ROW_INDEX])
local_plot_path = save_local_explanation_plot(
    local_df=local_df,
    predicted_price=local_prediction,
    base_value=base_value,
    output_path=OUTPUT_DIR / "local_explanation_example.png",
)
display(Image(filename=str(local_plot_path)))
local_df

## UI Shortlist

This shortlist is the bridge from interpretation to product design.
The goal is not to expose every model feature to the user, but to identify the small subset of inputs that are both influential and realistic for a user to provide or understand.

In [ ]:
suggested_ui_inputs = [
    feature
    for feature in importance_df["feature_name"].tolist()
    if feature not in {
        "model_total_registered",
        "model_median_vehicle_age",
        "model_mean_vehicle_age",
        "model_median_mileage",
        "model_mean_mileage",
        "model_median_engine_cc",
        "model_median_power_kw",
        "model_median_mass_kg",
        "brand_total_registered",
        "brand_median_vehicle_age",
        "brand_mean_vehicle_age",
        "brand_median_mileage",
        "brand_mean_mileage",
        "brand_median_engine_cc",
        "brand_median_power_kw",
        "brand_median_mass_kg",
        "model_share_of_market",
        "brand_share_of_market",
        "model_share_within_brand",
        "mileage_missing_flag",
    }
][:10]

pd.DataFrame({"suggested_ui_inputs": suggested_ui_inputs})

## Export Tables And Summary

This final cell writes the core outputs to `artifacts/random_forest_shap/` so they can be reused outside the notebook.

In [ ]:
summary = {
    "task_definition": "Tracked-snapshot price prediction for unseen marketplace listings using information available up to the current snapshot.",
    "sample_size": int(len(feature_frame)),
    "base_value": base_value,
    "top_features": importance_df.head(10).to_dict(orient="records"),
    "suggested_ui_inputs": suggested_ui_inputs,
    "local_example_prediction": local_prediction,
}

(OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
importance_df.to_csv(OUTPUT_DIR / "global_feature_importance.csv", index=False)
local_df.to_csv(OUTPUT_DIR / "local_explanation_example.csv", index=False)
feature_frame.to_csv(OUTPUT_DIR / "sampled_reference_rows.csv", index=False)

summary